# 08 - A simple hosted agent with Microsoft Agent Framework

A **hosted agent** is your own agent code running as a managed container in Microsoft Foundry. Foundry builds and runs it, gives it an identity, and exposes it through the **Responses API** - the same API you use to call a model, but the answer now comes from your code.

```text
main.py (Agent Framework)  --zip-->  Foundry builds + hosts it  <--responses.create()--  this notebook
   Agent + 1 function tool            create_version_from_code
   ResponsesHostServer
```

You will:

1. Write a tiny agent (`main.py`) with one function tool.
2. Zip it and create an agent version. No Docker and no `azd`; Foundry installs `requirements.txt` for you.
3. Wait until the version is active.
4. Call it with the Responses API.
5. Optionally delete it.

## Prerequisites

- The facilitator ran `scripts\deploy.ps1`, which creates the Foundry project and the `workshop-model` deployment, and gave you access (`grant-workshop-access.ps1`).
- You are signed in with `az login`.
- Hosted agents are a preview feature. If your region does not offer them, the create call in section 3 fails and the facilitator will tell you which project to use.
- The agent runs in Foundry, so you do **not** need `agent-framework` installed here. Only `azure-ai-projects` is used by the notebook.

In [ ]:
import os
import re
import time
import zipfile
from pathlib import Path
from uuid import UUID

from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_AI_PROJECT_ENDPOINT', 'AZURE_AI_MODEL_DEPLOYMENT_NAME')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Ask the facilitator; see .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
project_endpoint = os.environ['AZURE_AI_PROJECT_ENDPOINT'].rstrip('/')
model_name = os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')

agent_name = f'{prefix}-hosted-agent'
agent_folder = root / 'hosted-agent'

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential, allow_preview=True)
print('Project:', project_endpoint)
print('Agent name:', agent_name)

## 1. Write the agent

`main.py` is the whole agent. The parts that matter:

- `@tool` turns a normal Python function into a tool the model can call. The docstring and type hints become the tool description.
- `FoundryChatClient` talks to your model deployment. Inside Foundry, `FOUNDRY_PROJECT_ENDPOINT` and the agent's identity are provided by the platform, so there are no keys.
- `Agent` combines the model, instructions and tools.
- `ResponsesHostServer(agent).run()` serves the agent on the Responses protocol - this is what makes it *hosted*.

The cell writes `hosted-agent\main.py` and `hosted-agent\requirements.txt` next to the notebooks so you can open and edit them.

In [ ]:
main_py = """import os

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential

WORKSHOP_TIPS = {
    "web iq": "Web IQ grounds answers in the public web through one API key.",
    "foundry iq": "Foundry IQ puts several knowledge sources behind one knowledge base.",
    "indexer": "An indexer pulls documents from a data source into an AI Search index.",
}


@tool(approval_mode="never_require")
def get_workshop_tip(topic: str) -> str:
    \"\"\"Return a one-line tip about a workshop topic such as web iq, foundry iq or indexer.\"\"\"
    return WORKSHOP_TIPS.get(topic.lower(), "No tip for that topic. Try: " + ", ".join(WORKSHOP_TIPS))


client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)

agent = Agent(
    client=client,
    instructions="You are a friendly workshop assistant. Use get_workshop_tip when asked about a workshop topic.",
    tools=[get_workshop_tip],
    default_options={"store": False},
)

if __name__ == "__main__":
    ResponsesHostServer(agent).run()
"""

requirements_txt = """agent-framework-foundry
agent-framework-foundry-hosting>=1.0.0b261015
"""

agent_folder.mkdir(exist_ok=True)
(agent_folder / 'main.py').write_text(main_py, encoding='utf-8')
(agent_folder / 'requirements.txt').write_text(requirements_txt, encoding='utf-8')
compile(main_py, 'main.py', 'exec')
print('Wrote', [p.name for p in agent_folder.iterdir()], 'to', agent_folder)

## 2. Zip the code and create an agent version

An **agent version** is an immutable snapshot of the agent. `create_version_from_code` uploads the zip and describes how to run it:

| Setting | Meaning |
|---|---|
| `runtime` / `entry_point` | Python 3.13, started with `python main.py` |
| `dependency_resolution='remote_build'` | Foundry runs `pip install -r requirements.txt` for you |
| `protocol_versions` | The agent speaks the Responses protocol |
| `cpu` / `memory` | Container size |
| `environment_variables` | Passed to `main.py`. `FOUNDRY_*` names are reserved by the platform |

In [ ]:
from azure.ai.projects.models import CodeConfiguration, HostedAgentDefinition, ProtocolVersionRecord

zip_path = agent_folder / 'agent.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(agent_folder / 'main.py', 'main.py')
    bundle.write(agent_folder / 'requirements.txt', 'requirements.txt')

definition = HostedAgentDefinition(
    cpu='0.5',
    memory='1Gi',
    protocol_versions=[ProtocolVersionRecord(protocol='responses', version='2.0.0')],
    environment_variables={'AZURE_AI_MODEL_DEPLOYMENT_NAME': model_name},
    code_configuration=CodeConfiguration(
        runtime='python_3_13',
        entry_point=['main.py'],
        dependency_resolution='remote_build',
    ),
)

with open(zip_path, 'rb') as code_stream:
    version = project_client.agents.create_version_from_code(
        agent_name,
        definition=definition,
        code=code_stream,
        description='Workshop hosted agent with one function tool',
        metadata={'workshop': 'iq', 'attendee': prefix},
    )
print('Created version', version.version, 'status:', version.status)

## 3. Wait until the version is active

Foundry builds the container in the background. The cell polls every 15 seconds for up to 15 minutes and stops early if the build fails.

In [ ]:
deadline = time.time() + 15 * 60
while True:
    version = project_client.agents.get_version(agent_name, version.version)
    print('status:', version.status)
    if str(version.status).lower() == 'active':
        break
    if str(version.status).lower() == 'failed':
        raise RuntimeError(f'Build failed: {version}')
    if time.time() > deadline:
        raise TimeoutError('Still not active after 15 minutes.')
    time.sleep(15)
print('Agent is ready:', agent_name, 'version', version.version)

## 4. Call the agent

`get_openai_client(agent_name=...)` returns an OpenAI client already pointed at your agent's endpoint, so `responses.create` runs **your** `main.py`, not a bare model. Note there is no `model=` argument.

The first call creates a session; its ID comes back in `agent_session_id` and identifies the running container.

In [ ]:
openai_client = project_client.get_openai_client(agent_name=agent_name)

response = openai_client.responses.create(input='Give me a tip about foundry iq.')
print(response.output_text)
print('Session:', (response.model_extra or {}).get('agent_session_id'))

## 5. See the tool at work

The tip for `indexer` exists only inside `get_workshop_tip`, so the model can answer correctly only by calling your Python function. Edit `WORKSHOP_TIPS` in section 1, re-run sections 1-3, and ask again to publish a new version.

In [ ]:
response = openai_client.responses.create(input='What is the workshop tip for indexer? Quote it exactly.')
print(response.output_text)
for item in response.output:
    print('output item:', item.type)

## Cleanup

Hosted agents can consume quota while they exist. Set `CLEAN_UP = True` and run the cell to delete your agent and its versions. Other attendees' agents are not touched.

In [ ]:
CLEAN_UP = False

if CLEAN_UP:
    project_client.agents.delete(agent_name)
    print('Deleted', agent_name)
else:
    print('Cleanup skipped. Set CLEAN_UP = True to delete', agent_name)